In [ ]:
import gdown
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from MLP import *

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
Folder = 'data/'

In [ ]:
FILE_ID = '1BYk7DcE5uJuJq3cnOLrjlCbeq6J_V6k7'
FILE_NAME = 'NonLinear_data.npy'

In [ ]:
gdown.download(id= FILE_ID, output=Folder + FILE_NAME)

In [ ]:
data = np.load(Folder + FILE_NAME, allow_pickle=True).item()
data

In [ ]:
print(data.keys())

In [ ]:
X = data['X']
print(f"X = {X}")
y = data['labels']
print(f"y = {y}")

In [ ]:
# In ra các nhãn của bộ DL
np.unique(y)

# Mã hóa nhãn về dạng OneHot Encoding

In [ ]:
Encoder = OneHotEncoder(categories=[np.arange(3)],sparse_output=False)

In [ ]:
y_encoder = Encoder.fit_transform(y.reshape(-1,1))
y_encoder

# Chuẩn hóa các đặc trưng của bộ DL

In [ ]:
# Chuẩn hóa DL
Scaler = StandardScaler()
X = Scaler.fit_transform(X.copy())
X

In [ ]:
X = torch.tensor(X.copy(), dtype=torch.float32)
y = torch.tensor(y_encoder,dtype=torch.float32)

In [ ]:
# Chia bộ DL thành train set và test set
print(X.shape)
print(y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.1, random_state=42, shuffle=True)

In [ ]:
# Đưa dữ liệu lên GPU
X_train = X_train.to(device)
X_test = X_test.to(device)
y_train = y_train.to(device)
y_test = y_test.to(device)

# Xây dựng mô hình MultiLayer Perceptron Classifier

In [ ]:
BaseMLP??

In [ ]:
class MLPClassifier(BaseMLP):
    def __init__(self):
        super().__init__()
    def predict(self, X):
        logits = self.forward(X)
        return torch.argmax(logits, dim=1)
    def get_accuracy(self, logits, y):
        try:
            return torch.mean((torch.argmax(logits, dim = 1) == y).float())
        except:
            return torch.mean((torch.argmax(logits, dim = 1) == torch.argmax(y,dim=1)).float())
    def compute_loss(self, logits, y):
        return self.criterion(logits,y)

In [ ]:
model = MLPClassifier()

# add các Layers cho mô hình 

In [ ]:
input_dims = X_train.shape[1]
hidden_dims = 5
ouput_dims = y_train.shape[1]
model.Add_layer(
    nn.Linear(in_features=input_dims, out_features=hidden_dims)
)
# activation function
model.Add_layer(
    nn.GELU()
)
model.Add_layer(
    nn.Linear(in_features=hidden_dims, out_features=hidden_dims)
)
# activation function
model.Add_layer(
    nn.GELU()
)
model.Add_layer(
    nn.Linear(in_features=hidden_dims, out_features=ouput_dims)
)

In [ ]:
from torchsummary import summary
summary(model.model,input_size=(2,), batch_size=32, device='cpu')

In [ ]:
model.fit(X_train, y_train,batch_size=32, verbose=1, is_shuffle=True, optimizer='Adam', criterion='CE')

In [ ]:
epochs = range(1, len(model.Losses) + 1)

plt.figure(figsize=(12, 4))

# Loss
plt.subplot(1, 2, 1)
plt.plot(epochs, model.Losses, label="Train loss")
if getattr(model, "Val_Losses", None) and len(model.Val_Losses) > 0:
    plt.plot(epochs, model.Val_Losses, label="Val loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, model.Accuracies, label="Train accuracy")
if getattr(model, "Val_Accuracies", None) and len(model.Val_Accuracies) > 0:
    plt.plot(epochs, model.Val_Accuracies, label="Val accuracy")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
logist_test = model.forward(X_test).cpu()
y_true = y_test.cpu()
print(f"Độ chính xác của mô hình trên tập Test: {model.get_accuracy(logist_test, y_true):.4f}")
print(f"Loss của mô hình trên tập test: {model.compute_loss(logist_test, y_true):.4f}")